In [14]:
import numpy as np
import cv2

def extract_faces(image, mask):
  mask = (mask > 0.5).astype(np.uint8) * 255
  contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

  faces = []
  for contour in contours:
    x, y, w, h = cv2.boundingRect(contour)
    face = image[y:y+h, x:x+w]
    faces.append(face)

  return faces

In [10]:
def preprocess_face(face, target_size=(224, 224)):
    face = cv2.resize(face, target_size)
    face = face / 255.0
    return face

In [ ]:
from tensorflow.keras.models import load_model


segment = load_model('/content/vgg16_unet_1.h5')

In [5]:
classify = load_model('/content/vgg_charx224.h5')

In [11]:
def classify_faces(faces, classify):

    predictions = []
    for face in faces:
        face = preprocess_face(face)
        face = np.expand_dims(face, axis=0)
        pred = classify.predict(face)
        pred_class = np.argmax(pred, axis=1)[0]
        predictions.append(pred_class)
    return predictions

In [8]:
label_dict = {
    0: "Black-Widow",
    1: "Falcon",
    2: "Mary-Jane",
    3: "Ned",
    4: "Peter-Parker",
    5: "Spiderman",
    6: "Stanley",
    7: "Steve-Rogers",
    8: "Tony-Stark",
    9: "War-Machine",
    10: "Winter-Soldier",
}

def get_character_names(predictions, label_dict):

    return [label_dict[pred] for pred in predictions]

In [15]:
def process_image(image_path, segment, classify, label_dict):
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image_resized = cv2.resize(image, (512, 512))
    image_input = image_resized / 255.0
    image_input = np.expand_dims(image_input, axis=0)

    mask = segment.predict(image_input)[0]
    mask = np.squeeze(mask, axis=-1)

    faces = extract_faces(image_resized, mask)

    predictions = classify_faces(faces, classify)

    character_names = get_character_names(predictions, label_dict)

    return character_names

In [16]:

image_path = '/content/1.png'

character_names = process_image(image_path, segment, classify, label_dict)

print("Characters:", character_names)

1/1 ━━━━━━━━━━━━━━━━━━━━ 9s 9s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 746ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 546ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 552ms/step
Characters: ['Tony-Stark', 'War-Machine', 'Tony-Stark']
